---
title: ""
author: "Hoang Son Lai"
date: "05/15/2026"
categories: [Exploratory Data Analysis, Machine Learning]
format: 
 html:
  toc: true
  css: styles.css
  embed-resources: true
  code-fold: true
---

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

import xgboost as xgb
import shap

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('muted')

In [10]:
# Load Data
df = pd.read_csv('../data/melbourne_price_data_enriched.csv')
print(f"Shape: {df.shape}")
df.head(3)

Shape: (143974, 27)


,Property_ID,Status,Full_Address,Suburb,Postcode,Property_Type,Date,Beds,Baths,Car_Spaces,...,URL,Last_Updated,abs_median_income_weekly,abs_median_age,abs_population,crime_offence_count,crime_suburb_ref,crime_rate_per_100k,dist_nearest_train_km,Enriched_Date
0,2019150028,For Sale,"910/188 Ballarat Road, FOOTSCRAY VIC 3011",FOOTSCRAY,3011,Apartment / Unit / Flat,NaN,2,2,1,...,https://www.domain.com.au/910-188-ballarat-roa...,2026-03-14,981,34,22278,4471.0,Footscray,20069.1,0.019,2026-05-01
1,2010114712,Sold,"695 Mount Blackwood Road, KOROBEIT VIC 3341",KOROBEIT,3341,House,10 Dec 2012,3,1,2,...,https://www.domain.com.au/695-mount-blackwood-...,2026-03-17,861,43,1644,68.0,Myrniong,4136.3,11.159,2026-05-01
2,2011918466,Sold,"70 Lohs Lane, MYRNIONG VIC 3341",MYRNIONG,3341,House,18 May 2015,4,2,4,...,https://www.domain.com.au/70-lohs-lane-myrnion...,2026-03-17,861,43,1644,68.0,Myrniong,4136.3,10.663,2026-05-01


In [11]:
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])


Missing values:
Property_Type             76
Date                   30685
LandSize_sqm              76
Raw_Price                 76
Numeric_Price          13844
crime_offence_count       23
crime_suburb_ref          23
crime_rate_per_100k       23
dtype: int64


In [12]:
# Parse Date
df['Date_parsed'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df['Sale_Year']  = df['Date_parsed'].dt.year
df['Sale_Month'] = df['Date_parsed'].dt.month

n_valid = df['Date_parsed'].notna().sum()
print(f"Valid dates parsed: {n_valid:,} / {len(df):,} ({n_valid/len(df)*100:.1f}%)")

Valid dates parsed: 113,249 / 143,974 (78.7%)


In [13]:
# Numeric Price Cleaning
print("Price distribution before cleaning:")
print(df['Numeric_Price'].describe())

# Flag extreme low outliers — likely data entry errors, not real sales
LOW_THRESHOLD = 10_000
mask_low = df['Numeric_Price'] < LOW_THRESHOLD
print(f"\nRows with price < ${LOW_THRESHOLD:,}: {mask_low.sum()}")
df.loc[mask_low, 'Numeric_Price'] = np.nan

# High outlier
upper_cap = df['Numeric_Price'].quantile(0.995)
print(f"99.5th percentile cap: ${upper_cap:,.0f}")
# We will apply this cap later on the Sold training set only — not here globally.

Price distribution before cleaning:
count    1.301300e+05
mean     9.902159e+05
std      1.101320e+06
min      1.000000e+00
25%      6.200000e+05
50%      7.700000e+05
75%      1.100000e+06
max      1.400000e+08
Name: Numeric_Price, dtype: float64

Rows with price < $10,000: 43
99.5th percentile cap: $5,100,000


In [16]:
# Property Type Grouping
type_map = {
    'House':                          'House',
    'Townhouse':                      'Townhouse',
    'Apartment / Unit / Flat':        'Apartment',
    'Villa':                          'Villa',
    'Studio':                         'Apartment',
    'Penthouse':                      'Apartment',
    'Semi-Detached':                  'House',
    'Duplex':                         'House',
    'Terrace':                        'House',
    'Block of Units':                 'Apartment',
    'New House & Land':               'New House & Land',
    'New Home Designs':               'House',
    'New Apartments / Off the Plan':  'Apartment',
    'New land':                       'New Land',
    'Vacant land':                    'Vacant Land',
    'Acreage / Semi-Rural':           'Rural',
    'Rural':                          'Rural',
    'Rural Lifestyle':                'Rural',
    'Farm':                           'Rural',
    'Farmlet':                        'Rural',
    'Grazing':                        'Rural',
    'Livestock':                      'Rural',
    'Specialist Farm':                'Rural',
    'Development Site':               'Other',
    'Car Space':                      'Other',
    'Unknown':                        'Other',
}

df['Property_Group'] = df['Property_Type'].map(type_map).fillna('Other')
print(df['Property_Group'].value_counts())

Property_Group
House               94826
Apartment           17806
Townhouse           10543
Vacant Land         10266
New House & Land     7675
Rural                2179
Villa                 311
Other                 201
New Land              167
Name: count, dtype: int64


In [8]:
# Flag vacant / land-type properties
df['Is_Land'] = df['Property_Group'].isin(['Land', 'Rural']).astype(int)

# LandSize = 0 for apartments is plausible (strata title, no individual land)
# LandSize = 0 for houses/townhouses is suspicious — set to NaN
mask_land_zero = (df['LandSize_sqm'] == 0) & (~df['Is_Land'].astype(bool)) & (df['Property_Group'].isin(['House', 'Townhouse']))
print(f"House/Townhouse rows with LandSize = 0 set to NaN: {mask_land_zero.sum():,}")
df.loc[mask_land_zero, 'LandSize_sqm'] = np.nan

print(f"\nLandSize missing after fix: {df['LandSize_sqm'].isna().sum():,}")

House/Townhouse rows with LandSize = 0 set to NaN: 27,530

LandSize missing after fix: 27,606
